In [17]:

import pandas as pd
import numpy as np

url="https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"

df = pd.read_csv(url)

print(df.shape)
df.head()


(5000, 9)


,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [18]:

df.isnull().sum()


lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [19]:

missing = df.isnull().sum()
missing[missing > 0]


lead_source          148
industry             240
employment_status    194
location             208
annual_income        369
lead_score            35
dtype: int64

In [20]:

categorical = df.select_dtypes(include=["object"]).columns
numerical = df.select_dtypes(include=["number"]).columns

df[categorical] = df[categorical].fillna("NA")
df[numerical] = df[numerical].fillna(0.0)


C:\Users\EBIHA10\AppData\Local\Temp\ipykernel_31560\1591270590.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical = df.select_dtypes(include=["object"]).columns


In [21]:

df.isnull().sum().sum()


np.int64(0)

In [22]:

from sklearn.metrics import roc_auc_score

numerical_features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

for feature in numerical_features:
    auc = roc_auc_score(df["converted"], df[feature])
    print(f"{feature}: {auc:.4f}")


lead_score: 0.7872
number_of_courses_viewed: 0.7228
interaction_count: 0.7624
annual_income: 0.6098


In [23]:

from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)


In [24]:

from sklearn.metrics import roc_auc_score

features = [
    "lead_score",
    "number_of_courses_viewed",
    "interaction_count",
    "annual_income"
]

for feature in features:
    score = roc_auc_score(df_train["converted"], df_train[feature])

    if score < 0.5:
        score = roc_auc_score(df_train["converted"], -df_train[feature])

    print(f"{feature}: {score:.4f}")


lead_score: 0.7882
number_of_courses_viewed: 0.7230
interaction_count: 0.7649
annual_income: 0.6082


In [25]:

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y_train = df_train["converted"].values
y_val = df_val["converted"].values

dv = DictVectorizer(sparse=True)

train_dicts = df_train.drop(columns=["converted"]).to_dict(orient="records")
val_dicts = df_val.drop(columns=["converted"]).to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver="liblinear",
    C=1.0,
    max_iter=1000
)
import numpy as np
from scipy.sparse import csr_matrix

X_train.indices = X_train.indices.astype(np.int32)
X_train.indptr = X_train.indptr.astype(np.int32)

X_val.indices = X_val.indices.astype(np.int32)
X_val.indptr = X_val.indptr.astype(np.int32)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]

auc = roc_auc_score(y_val, y_pred)
print(f"Validation AUC: {auc:.3f}")


Validation AUC: 0.732


In [26]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_val, y_pred)
print(round(auc, 3))

0.732


In [27]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.0, 1.01, 0.01)

results = []

for t in thresholds:
    y_pred_binary = (y_pred >= t).astype(int)

    precision = precision_score(y_val, y_pred_binary, zero_division=0)
    recall = recall_score(y_val, y_pred_binary, zero_division=0)

    if precision != 0 or recall != 0:
        results.append((t, precision, recall, abs(precision - recall)))

best = min(results, key=lambda x: x[3])

print(f"Threshold: {best[0]:.2f}")
print(f"Precision: {best[1]:.3f}")
print(f"Recall:    {best[2]:.3f}")

Threshold: 0.63
Precision: 0.733
Recall:    0.725


In [28]:
import numpy as np
from sklearn.metrics import precision_score, recall_score

thresholds = np.arange(0.0, 1.01, 0.01)
best_f1 = 0
best_threshold = 0

for t in thresholds:
    y_pred_binary = (y_pred >= t).astype(int)

    p = precision_score(y_val, y_pred_binary, zero_division=0)
    r = recall_score(y_val, y_pred_binary, zero_division=0)

    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

print(f"Best threshold: {best_threshold:.2f}")
print(f"Max F1: {best_f1:.3f}")

Best threshold: 0.41
Max F1: 0.758


In [ ]:
from sklearn.model_selection import KFold
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx].copy()
    df_val = df_full_train.iloc[val_idx].copy()

    y_train = df_train['converted'].values
    y_val = df_val['converted'].values

    train_dicts = df_train[features].to_dict(orient='records')
    val_dicts = df_val[features].to_dict(orient='records')

    dv = DictVectorizer(sparse=True)
    X_train = dv.fit_transform(train_dicts)
    X_val = dv.transform(val_dicts)

    X_train.indices = X_train.indices.astype(np.int32)
    X_train.indptr = X_train.indptr.astype(np.int32)
    X_val.indices = X_val.indices.astype(np.int32)
    X_val.indptr = X_val.indptr.astype(np.int32)

    model = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000
    )

    model.fit(X_train, y_train)
    y_pred = model.predict_proba(X_val)[:, 1]

    scores.append(roc_auc_score(y_val, y_pred))

print("Scores:", scores)
print("Mean AUC:", np.mean(scores))
print("Standard deviation:", np.std(scores))

Scores: [0.8087112822981866, 0.8048844020610042, 0.9999871279991761, 1.0, 0.9999806221579165]
Mean AUC: 0.9227126869032567
Standard deviation: 0.09465181117307259


In [42]:
categorical = [
    'lead_source',
    'industry',
    'employment_status',
    'location'
]

numerical = [
    'annual_income',
    'number_of_courses_viewed',
    'interaction_count',
    'lead_score'
]

features = list(categorical) + list(numerical)

print("Features:", features)
print("Target in features:", 'converted' in features)

Features: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
Target in features: False


In [40]:
print(df_full_train['converted'].value_counts(dropna=False))
print(df_full_train['converted'].value_counts(normalize=True, dropna=False))
print(df_full_train['converted'].dtype)

converted
1    2315
0    1685
Name: count, dtype: int64
converted
1    0.57875
0    0.42125
Name: proportion, dtype: float64
int64


In [41]:
print("Categorical:", list(categorical))
print("Numerical:", list(numerical))
print("Features:", features)

Categorical: ['lead_source', 'industry', 'employment_status', 'location']
Numerical: ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
Features: ['lead_source', 'industry', 'employment_status', 'location', 'annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
